# DINOv3 vs EUPE 鈥?STM 5-Class Joint Segmentation

**Strategy**: modulation + defect data trained together. Model learns that modulation != bright_defect.

| class_id | name | source | shape |
|----------|------|--------|-------|
| 0 | background | 鈥?| 鈥?|
| 1 | modulation_region | png-modulation | polygon |
| 2 | sqrt2_modulation_region | png-modulation | polygon |
| 3 | dark_defect | png-defect | rectangle |
| 4 | bright_defect | png-defect | rectangle |

**Key**: modulation images have classes 1/2 but not 3/4; defect images have 3/4 but not 1/2.
`ignore_index=-1` in CrossEntropyLoss skips absent classes per image.

| Encoder | embed_dim | depth | mode |
|---------|-----------|-------|------|
| DINOv3 ViT-L | 1024 | 24 | head_only |
| EUPE ViT-B | 768 | 12 | head_only |

**Total**: 8 labeled images (4 mod + 4 defect)
**Head**: DINOv3LinearSegmentationHead
**Loss**: DiceCELoss (CE + 0.5xDice, ignore_index=-1)

In [ ]:
from __future__ import annotations
import json, random, sys
from pathlib import Path
from typing import Any
import numpy as np
import torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset, ConcatDataset
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from PIL import Image as PILImage, ImageDraw

def find_repo_root(start):
    for p in [start.resolve(), *start.resolve().parents]:
        if (p/'src'/'lumen').exists(): return p
    raise RuntimeError('repo root not found')

REPO_ROOT = find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT/'src'))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

STM = REPO_ROOT/'data'/'stm_dataset'/'FeTe-sxm'
MOD_DIR = STM/'png-modulation'
DEF_DIR = STM/'png-defect'
DINOV3_PATH = REPO_ROOT/'checkpoints'/'dinov3-vitl16-pretrain-lvd1689m'
EUPE_PATH   = REPO_ROOT/'checkpoints'/'EUPE-ViT-B.pt'

CLASS_NAMES = ['background','modulation_region','sqrt2_modulation_region','dark_defect','bright_defect']
NUM_CLASSES = len(CLASS_NAMES)
MOD_LABEL_TO_ID = {'modulation_region':1, 'sqrt2_modulation_region':2}
DEF_LABEL_TO_ID = {'dark_defect':3, 'bright_defect':4}

IMAGE_SIZE = 512; BATCH_SIZE = 2; EPOCHS = 80
HEAD_LR = 5e-4; DICE_WEIGHT = 0.5; WEIGHT_DECAY = 1e-4; SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


# FFT modulation map toggle
FFT_ENABLE = True  # False=original 1ch, True=3ch [gray, gray, FFT_map]

# -- FFT bandpass modulation map --
from scipy.ndimage import gaussian_filter
def compute_fft_modulation_map(image, low_cut=8, high_cut=None, sigma=2.0):
    h, w = image.shape
    f = np.fft.fft2(image); fshift = np.fft.fftshift(f)
    cy, cx = h//2, w//2; y, x = np.ogrid[:h,:w]
    dist = np.sqrt((x-cx)**2 + (y-cy)**2)
    if high_cut is None: high_cut = min(h,w)//4
    bandpass = np.ones((h,w))
    bandpass[dist < low_cut] = 0; bandpass[dist > high_cut] = 0
    bandpass = gaussian_filter(bandpass.astype(float), sigma=sigma)
    f_filtered = fshift * bandpass; f_ishift = np.fft.ifftshift(f_filtered)
    img_mod = np.abs(np.fft.ifft2(f_ishift))
    lo, hi = np.percentile(img_mod, (1,99))
    img_mod = np.clip((img_mod-lo)/(hi-lo+1e-8), 0, 1)
    return img_mod.astype(np.float32)

print(f'FFT_ENABLE={FFT_ENABLE} ({"3ch pseudo-RGB" if FFT_ENABLE else "1ch grayscale"})')

print(f'Device: {DEVICE}')
print(f'MOD_DIR: {MOD_DIR.exists()} | DEF_DIR: {DEF_DIR.exists()}')
print(f'DINOv3: {DINOV3_PATH.exists()} | EUPE: {EUPE_PATH.exists()}')

## 1. Dataset: 4 modulation + 4 defect -> ConcatDataset

In [ ]:
class LabelMeSegDataset(Dataset):
    def __init__(self, data_dir, label_to_id, image_size=512, augment=False, use_fft=True):
        super().__init__()
        self.data_dir = Path(data_dir); self.label_to_id = label_to_id
        self.image_size = image_size; self.augment = augment
        self.use_fft = use_fft  # only compute FFT map for modulation (not defect)
        self.samples = []
        for jf in sorted(self.data_dir.glob('*.json')):
            png = jf.with_suffix('.png')
            if not png.exists(): continue
            ann = json.loads(jf.read_text(encoding='utf-8'))
            shapes = []
            for s in ann.get('shapes',[]):
                cid = self.label_to_id.get(s['label'])
                if cid is None: continue
                shapes.append({'label_id':cid, 'points':s['points'],
                               'shape_type':s.get('shape_type','polygon')})
            if not shapes: continue
            self.samples.append({'stem':jf.stem, 'png_path':png, 'shapes':shapes,
                                 'img_h':ann.get('imageHeight',0), 'img_w':ann.get('imageWidth',0)})
        self.stems = [s['stem'] for s in self.samples]

    def __len__(self): return len(self.samples)

    def _render_mask(self, shapes, H, W):
        mask = PILImage.new('L',(W,H),0)
        d = ImageDraw.Draw(mask)
        for s in shapes:
            pts, cid, st = s['points'], s['label_id'], s['shape_type']
            if st == 'polygon' and len(pts)>=3:
                d.polygon([(float(p[0]),float(p[1])) for p in pts], fill=cid)
            elif st == 'rectangle' and len(pts)>=2:
                x0,y0 = float(pts[0][0]),float(pts[0][1])
                x1,y1 = float(pts[1][0]),float(pts[1][1])
                d.rectangle([min(x0,x1),min(y0,y1),max(x0,x1),max(y0,y1)], fill=cid)
        return np.asarray(mask, dtype=np.int64)

    def _load_and_resize(self, sample):
        img = PILImage.open(sample['png_path']).convert('L')
        W,H = img.size; S = self.image_size
        mask = self._render_mask(sample['shapes'], H, W)
        img_arr = np.asarray(img.resize((S,S),PILImage.BILINEAR),np.float32)/255.
        mask_pil = PILImage.fromarray(mask.astype(np.int32),mode='I')
        return img_arr, np.asarray(mask_pil.resize((S,S),PILImage.NEAREST),np.int64)

    def _augment(self, image, mask):
        if not self.augment: return image, mask
        if random.random()<0.5: image=image[:,::-1].copy(); mask=mask[:,::-1].copy()
        if random.random()<0.5: image=image[::-1,:].copy(); mask=mask[::-1,:].copy()
        k = random.randint(0,3)
        if k: image=np.rot90(image,k).copy(); mask=np.rot90(mask,k).copy()
        if random.random()<0.5:
            image = np.clip(image*(1+(random.random()-.5)*.4)+(random.random()-.5)*.2,0,1)
        return image, mask

    def __getitem__(self, index):
        s = self.samples[index]; image, mask = self._load_and_resize(s)
        image, mask = self._augment(image, mask)
        if FFT_ENABLE:
            # All images: 3ch [gray, gray, fft_map] for consistent channel semantics
            # Modulation: fft high -> modulation; Defect: fft low -> defect
            fft_map = compute_fft_modulation_map(image)
            image_t = torch.from_numpy(np.stack([image, image, fft_map], axis=0).astype(np.float32))
        else:
            image_t = torch.from_numpy(image).unsqueeze(0)
        return {'image': image_t, 'mask': torch.from_numpy(mask), 'stem': s['stem'],
                'source': str(self.data_dir.name)}

mod_ds = LabelMeSegDataset(MOD_DIR, MOD_LABEL_TO_ID, IMAGE_SIZE, augment=False)
def_ds = LabelMeSegDataset(DEF_DIR, DEF_LABEL_TO_ID, IMAGE_SIZE, augment=False)

dataset = ConcatDataset([mod_ds, def_ds])
print(f'Modulation: {len(mod_ds)} samples | Defect: {len(def_ds)} samples')
print(f'Total: {len(dataset)} samples')
print()
print('Modulation:')
for s in mod_ds.samples:
    labs = {}
    for sh in s['shapes']:
        n = CLASS_NAMES[sh['label_id']]; labs[n]=labs.get(n,0)+1
    print(f'  {s["stem"]}: {len(s["shapes"])} shapes, {labs}')
print('Defect:')
for s in def_ds.samples:
    labs = {}
    for sh in s['shapes']:
        n = CLASS_NAMES[sh['label_id']]; labs[n]=labs.get(n,0)+1
    print(f'  {s["stem"]}: {len(s["shapes"])} shapes, {labs}')

## 2. Annotation viz: mask overlay

In [ ]:
def mask_to_rgb5(mask):
    rgb = np.zeros((*mask.shape,3))
    rgb[mask==1] = [0,1,1]
    rgb[mask==2] = [1,0,1]
    rgb[mask==3] = [1,0,0]
    rgb[mask==4] = [1,1,0]
    return rgb

n = len(dataset)
fig, axes = plt.subplots(2, n, figsize=(3*n, 8))
if n == 1: axes = axes[:,np.newaxis]
for i in range(n):
    s = dataset[i]; img = s['image'][0].numpy(); m = s['mask'].numpy()
    axes[0,i].imshow(img, cmap='gray')
    axes[0,i].set_title(f"{s['stem']}\n{s['source']}", fontsize=8); axes[0,i].axis('off')
    axes[1,i].imshow(img, cmap='gray'); axes[1,i].imshow(mask_to_rgb5(m), alpha=0.5)
    counts = ', '.join(f'{CLASS_NAMES[c][:4]}={(m==c).sum()}' for c in range(1,5) if (m==c).any())
    axes[1,i].set_title(counts, fontsize=7); axes[1,i].axis('off')
leg = [Line2D([0],[0],color=c,lw=4,label=CLASS_NAMES[i]) for i,c in enumerate(['cyan','magenta','red','yellow'],1)]
fig.legend(handles=leg, loc='upper center', ncol=4)
plt.tight_layout(); plt.show()

## 3. Train/val split

1 mod + 1 defect in val; 3 mod + 3 defect in train.

In [ ]:
mod_idx = list(range(len(mod_ds)))
def_idx = list(range(len(mod_ds), len(dataset)))
random.Random(SEED).shuffle(mod_idx)
random.Random(SEED).shuffle(def_idx)

val_indices = sorted([mod_idx[0], def_idx[0]])
train_indices = sorted(mod_idx[1:] + def_idx[1:])

val_dataset = Subset(dataset, val_indices)
print(f'Train: {len(train_indices)}, Val: {len(val_indices)}')
print(f'Train: {[dataset[i]["stem"]+"("+dataset[i]["source"]+")" for i in train_indices]}')
print(f'Val:   {[dataset[i]["stem"]+"("+dataset[i]["source"]+")" for i in val_indices]}')

mod_aug = LabelMeSegDataset(MOD_DIR, MOD_LABEL_TO_ID, IMAGE_SIZE, augment=True)
def_aug = LabelMeSegDataset(DEF_DIR, DEF_LABEL_TO_ID, IMAGE_SIZE, augment=True)
dataset_aug = ConcatDataset([mod_aug, def_aug])
train_dataset = Subset(dataset_aug, train_indices)

counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for i in train_indices:
    flat = dataset[i]['mask'].numpy().ravel()
    for c in range(NUM_CLASSES): counts[c] += int((flat == c).sum())
freqs = counts / counts.sum()
inv = 1.0 / np.clip(freqs, 1e-6, None)
class_weights = torch.tensor(inv / inv.sum() * NUM_CLASSES, dtype=torch.float32).to(DEVICE)

for cid, name in enumerate(CLASS_NAMES):
    print(f'  {cid} ({name}): {counts[cid]} px, weight={class_weights[cid].item():.4f}')

## 4. Build DINOv3 + EUPE trainers

In [ ]:
from lumen.models import DINOv3Encoder
from lumen.models.eupe import EUPEEncoder
from lumen.training.downstream import SegmentationTrainer

class DiceCELoss(nn.Module):
    def __init__(self, ce_weight=None, dice_weight=0.5, smooth=1.0):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=ce_weight, ignore_index=-1)
        self.dice_weight = dice_weight; self.smooth = smooth
    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)
        probs = torch.softmax(logits,1); C = logits.shape[1]
        targets_oh = F.one_hot(targets, C).permute(0,3,1,2).float()
        dice = 0.0; n = 0
        for c in range(1,C):
            inter = (probs[:,c]*targets_oh[:,c]).sum()
            union = probs[:,c].sum()+targets_oh[:,c].sum()
            if union>0: dice += 1-(2*inter+self.smooth)/(union+self.smooth); n += 1
        return ce_loss + self.dice_weight*dice/max(n,1)

cweight = class_weights.detach().cpu()

print('Loading DINOv3...')
dino_encoder = DINOv3Encoder(model_dir=DINOV3_PATH, device=DEVICE, local_files_only=True)
dino_trainer = SegmentationTrainer(encoder=dino_encoder, num_classes=NUM_CLASSES,
    trainability='head_only', segmentation_head_name='dinov3-linear',
    segmentation_loss='ce', scheduler_name='cosine', scheduler_t_max=EPOCHS,
    lr=HEAD_LR, weight_decay=WEIGHT_DECAY).to(DEVICE)
dino_trainer.criterion = DiceCELoss(ce_weight=cweight, dice_weight=DICE_WEIGHT).to(DEVICE)
print(f'  DINOv3: trainable={sum(p.numel() for p in dino_trainer.parameters() if p.requires_grad):,}')

print('Loading EUPE...')
eupe_encoder = EUPEEncoder.from_pretrained(checkpoint_path=EUPE_PATH, device=DEVICE, strict=False)
eupe_encoder.auto_convert_input_channels = True
eupe_trainer = SegmentationTrainer(encoder=eupe_encoder, num_classes=NUM_CLASSES,
    trainability='head_only', segmentation_head_name='dinov3-linear',
    segmentation_loss='ce', scheduler_name='cosine', scheduler_t_max=EPOCHS,
    lr=HEAD_LR, weight_decay=WEIGHT_DECAY).to(DEVICE)
eupe_trainer.criterion = DiceCELoss(ce_weight=cweight, dice_weight=DICE_WEIGHT).to(DEVICE)
print(f'  EUPE: trainable={sum(p.numel() for p in eupe_trainer.parameters() if p.requires_grad):,}')
print(f'\nLoss: DiceCELoss(dice_weight={DICE_WEIGHT}, ignore_index=-1), Both: head_only')

## 5. Training loop

In [ ]:
def collate(batch):
    return {'image': torch.stack([b['image'] for b in batch]),
            'mask': torch.stack([b['mask'] for b in batch]),
            'stem': [b['stem'] for b in batch]}

train_loader = DataLoader(train_dataset, BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader = DataLoader(val_dataset, BATCH_SIZE, shuffle=False, collate_fn=collate)

@torch.no_grad()
def calc_miou(logits, mask):
    mask = mask.to(logits.device)
    pred = logits.argmax(1); ious = []
    for c in range(NUM_CLASSES):
        p=pred==c; g=mask==c
        inter=(p&g).sum().item(); union=(p|g).sum().item()
        ious.append(inter/union if union>0 else float('nan'))
    valid = [v for v in ious if not np.isnan(v)]
    return {'miou': np.mean(valid) if valid else 0.0, 'ious': ious}

CKPT_DIR = REPO_ROOT/'artifacts'/'joint5_seg'; CKPT_DIR.mkdir(parents=True, exist_ok=True)

def train_one(name, trainer):
    history = {'train_loss':[], 'val_loss':[], 'val_miou':[]}
    best_miou = -1.0; BEST = CKPT_DIR/f'best_{name}.pt'
    print(f'\nTraining: {name}')
    for epoch in range(EPOCHS):
        trainer.train(); losses = []
        for b in train_loader:
            b = {k:v.to(DEVICE) if torch.is_tensor(v) else v for k,v in b.items()}
            losses.append(trainer.train_step(b)['loss'])
        t_loss = float(np.mean(losses))
        trainer.eval(); v_losses, v_mious = [], []
        with torch.no_grad():
            for b in val_loader:
                b = {k:v.to(DEVICE) if torch.is_tensor(v) else v for k,v in b.items()}
                logits = trainer(b['image'])
                v_losses.append(trainer.compute_loss(logits, b['mask']).item())
                v_mious.append(calc_miou(logits, b['mask'])['miou'])
        v_loss = float(np.mean(v_losses)); v_miou = float(np.mean(v_mious))
        history['train_loss'].append(t_loss); history['val_loss'].append(v_loss)
        history['val_miou'].append(v_miou)
        if v_miou > best_miou:
            best_miou = v_miou
            torch.save({'head':trainer.head.state_dict(),'epoch':epoch,'miou':v_miou}, BEST)
        if (epoch+1)%20==0 or epoch<5:
            m=' [BEST]' if v_miou>=best_miou else ''
            print(f'  [{name}] {epoch+1:3d}/{EPOCHS}: train={t_loss:.4f} val={v_loss:.4f} mIoU={v_miou:.4f}{m}')
    print(f'\nDone. Best mIoU={best_miou:.4f}')
    return history, best_miou, BEST

dino_h, dino_miou, DINO_BEST = train_one('DINOv3_ViT-L', dino_trainer)
eupe_h, eupe_miou, EUPE_BEST = train_one('EUPE_ViT-B', eupe_trainer)
print(f'\nDINOv3: {dino_miou:.4f} | EUPE: {eupe_miou:.4f} | Best: {"EUPE" if eupe_miou>dino_miou else "DINOv3"}')

## 6. Loss curves + val predictions + per-class IoU

In [ ]:
fig, (ax1,ax2) = plt.subplots(1,2,figsize=(14,5))
for h,c,n in [(dino_h,'C0','DINOv3'),(eupe_h,'C1','EUPE')]:
    ax1.plot(h['train_loss'],c+'-',alpha=.4,label=f'{n} train')
    ax1.plot(h['val_loss'],c+'-',lw=2,label=f'{n} val')
    ax2.plot(h['val_miou'],c+'-',lw=2,label=f'{n} best={max(h["val_miou"]):.4f}')
ax1.set_xlabel('Epoch');ax1.set_ylabel('Loss');ax1.set_title('DiceCELoss');ax1.legend();ax1.grid(alpha=.3)
ax2.set_xlabel('Epoch');ax2.set_ylabel('mIoU');ax2.set_title('Validation mIoU');ax2.legend();ax2.grid(alpha=.3)
plt.tight_layout();plt.show()

dino_trainer.head.load_state_dict(torch.load(DINO_BEST,map_location=DEVICE,weights_only=True)['head']);dino_trainer.eval()
eupe_trainer.head.load_state_dict(torch.load(EUPE_BEST,map_location=DEVICE,weights_only=True)['head']);eupe_trainer.eval()

@torch.no_grad()
def infer(tr, img_t): return tr(img_t).argmax(1)[0].cpu().numpy()

n_show = len(val_indices)
fig, axes = plt.subplots(n_show, 4, figsize=(16, 4*n_show))
if n_show == 1: axes = axes.reshape(1,-1)

for row, idx in enumerate(val_indices):
    s = dataset[idx]; img = s['image'][0].numpy()
    img_t = s['image'].unsqueeze(0).to(DEVICE); gt = s['mask'].numpy()
    dp = infer(dino_trainer, img_t); ep = infer(eupe_trainer, img_t)
    
    def iou_per_cls(pred):
        return {c: ((pred==c)&(gt==c)).sum()/((pred==c)|(gt==c)).sum() for c in range(1,5) if ((pred==c)|(gt==c)).sum()>0}
    
    diou = iou_per_cls(dp); eiou = iou_per_cls(ep)
    d_str = ', '.join(f'{CLASS_NAMES[c][:4]}={diou[c]:.2f}' for c in diou)
    e_str = ', '.join(f'{CLASS_NAMES[c][:4]}={eiou[c]:.2f}' for c in eiou)
    
    titles = [f"{s['stem']}", 'GT', f'DINOv3\n{d_str}', f'EUPE\n{e_str}']
    masks = [None, gt, dp, ep]
    for c, (title, mask) in enumerate(zip(titles, masks)):
        axes[row,c].imshow(img, cmap='gray')
        if mask is not None: axes[row,c].imshow(mask_to_rgb5(mask), alpha=0.5)
        axes[row,c].set_title(title, fontsize=9); axes[row,c].axis('off')

winner = 'EUPE' if eupe_miou>dino_miou else 'DINOv3'
fig.suptitle(f'5-Class Joint Seg 鈥?{winner} wins ({max(dino_miou,eupe_miou):.4f})', fontsize=14)
plt.tight_layout();plt.show()

print(f"\n{'class':25s} {'DINOv3':>8s} {'EUPE':>8s}")
print('-'*45)
for c in range(5):
    dv = [v for v in [calc_miou(dino_trainer(dataset[i]['image'].unsqueeze(0).to(DEVICE)),dataset[i]['mask'].unsqueeze(0))['ious'][c] for i in val_indices] if not np.isnan(v)]
    ev = [v for v in [calc_miou(eupe_trainer(dataset[i]['image'].unsqueeze(0).to(DEVICE)),dataset[i]['mask'].unsqueeze(0))['ious'][c] for i in val_indices] if not np.isnan(v)]
    print(f'{CLASS_NAMES[c]:25s} {np.mean(dv) if dv else 0:8.4f} {np.mean(ev) if ev else 0:8.4f}')

## 7. 16 unlabeled images inference

In [ ]:
lbl_stems = {dataset[i]['stem'] for i in range(len(dataset))}
unlabeled = sorted([p for d in [MOD_DIR, DEF_DIR] for p in d.glob('*.png') if p.stem not in lbl_stems])
print(f'Unlabeled: {len(unlabeled)}')

def preprocess(img_np):
    """Convert 1ch grayscale -> 3ch [gray, gray, fft_map] for ALL images."""
    if FFT_ENABLE:
        fft_map = compute_fft_modulation_map(img_np)
        img_3ch = np.stack([img_np, img_np, fft_map], axis=0)
        return torch.from_numpy(img_3ch.astype(np.float32))
    return torch.from_numpy(img_np.astype(np.float32)).unsqueeze(0)

def load_infer(png):
    img = np.asarray(PILImage.open(png).convert('L').resize((IMAGE_SIZE,IMAGE_SIZE),PILImage.BILINEAR),np.float32)/255.
    t = preprocess(img).unsqueeze(0).to(DEVICE)
    return img, infer(dino_trainer,t), infer(eupe_trainer,t)

results = [(p.stem, *load_infer(p)) for p in unlabeled]

for page, start in enumerate([0,8]):
    batch = results[start:start+8]; n = len(batch)
    fig, axes = plt.subplots(n,3,figsize=(15,3.5*n))
    if n==1: axes=axes.reshape(1,-1)
    for row,(stem,img,dp,ep) in enumerate(batch):
        axes[row,0].imshow(img,cmap='gray'); axes[row,0].set_title(stem,fontsize=8); axes[row,0].axis('off')
        for col,(mask,name) in enumerate([(dp,f'DINOv3'),(ep,f'EUPE')]):
            axes[row,col+1].imshow(img,cmap='gray'); axes[row,col+1].imshow(mask_to_rgb5(mask),alpha=0.5)
            counts = ','.join(f'{CLASS_NAMES[c][:4]}={(mask==c).sum()}' for c in range(1,5) if (mask==c).any()) or 'all bg'
            axes[row,col+1].set_title(f'{name}\n{counts}',fontsize=7); axes[row,col+1].axis('off')
    fig.suptitle(f'5-Class Joint Inference 鈥?Page {page+1}/2',fontsize=13)
    plt.tight_layout();plt.show()

hdr = f"{'stem':12s} {'mod':>6s} {'sq2':>6s} {'dark':>6s} {'br':>6s} | {'mod':>6s} {'sq2':>6s} {'dark':>6s} {'br':>6s}"
print(hdr)
print(f"{'':12s} {'DINOv3':>24s}  | {'EUPE':>24s}")
print('-'*65)
for stem,img,dp,ep in results:
    d = [int((dp==c).sum()) for c in range(1,5)]
    e = [int((ep==c).sum()) for c in range(1,5)]
    print(f'{stem:12s} {d[0]:6d} {d[1]:6d} {d[2]:6d} {d[3]:6d} | {e[0]:6d} {e[1]:6d} {e[2]:6d} {e[3]:6d}')

## 8. Custom pick visualization

Change `PICK_STEMS` to select images to view.

In [ ]:
# Custom pick visualization - edit PICK_STEMS to choose images
# Right-click figure -> Save Image As, or set SAVE_FIG=True
SAVE_FIG = False
PICK_STEMS = ['FeTe_0001','FeTe_0003','FeTe_0010','FeTe_0017','FeTe_0020']

# Load best heads
dino_ckpt = torch.load(DINO_BEST, map_location=DEVICE, weights_only=True)
dino_trainer.head.load_state_dict(dino_ckpt.get("head_state_dict", dino_ckpt.get("head", dino_ckpt))); dino_trainer.eval()
eupe_ckpt = torch.load(EUPE_BEST, map_location=DEVICE, weights_only=True)
eupe_trainer.head.load_state_dict(eupe_ckpt.get("head_state_dict", eupe_ckpt.get("head", eupe_ckpt))); eupe_trainer.eval()

# Build lookup: all PNGs in both data dirs
all_pngs = {}
for d in [MOD_DIR, DEF_DIR]:
    for p in d.glob("*.png"):
        if p.stem not in all_pngs:
            all_pngs[p.stem] = p
_labeled_stems = {s['stem'] for ds in [mod_ds, def_ds] for s in ds.samples}

@torch.no_grad()
def load_and_predict(path):
    img = np.asarray(PILImage.open(path).convert("L").resize((IMAGE_SIZE, IMAGE_SIZE), PILImage.BILINEAR), dtype=np.float32) / 255.
    t = torch.from_numpy(img).unsqueeze(0).unsqueeze(0).to(DEVICE)
    d_pred = dino_trainer(t).argmax(dim=1)[0].cpu().numpy()
    e_pred = eupe_trainer(t).argmax(dim=1)[0].cpu().numpy()
    return img, d_pred, e_pred

valid = [(s, all_pngs[s]) for s in PICK_STEMS if s in all_pngs]
for s in PICK_STEMS:
    if s not in all_pngs:
        print(f"Warning: {s} not found, skipping")

n = len(valid)
if n == 0:
    print("No valid files found for PICK_STEMS")
else:
    fig, axes = plt.subplots(n, 3, figsize=(15, 4.5 * n))
    if n == 1: axes = axes.reshape(1, -1)
    for row, (stem, png_path) in enumerate(valid):
        has_label = stem in _labeled_stems
        tag = " [labeled]" if has_label else ""
        img, d_pred, e_pred = load_and_predict(png_path)
        # Col 0: input
        axes[row, 0].imshow(img, cmap="gray")
        axes[row, 0].set_title(f"{stem}{tag}", fontsize=10)
        axes[row, 0].axis("off")
        # Col 1: DINOv3 鈥?5-class overlay
        axes[row, 1].imshow(img, cmap="gray")
        axes[row, 1].imshow(mask_to_rgb5(d_pred), alpha=0.5)
        d_counts = ','.join(f'{CLASS_NAMES[c][:4]}={(d_pred==c).sum()}' for c in range(1,5) if (d_pred==c).any()) or 'all bg'
        axes[row, 1].set_title(f"DINOv3\n{d_counts}", fontsize=8)
        axes[row, 1].axis("off")
        # Col 2: EUPE 鈥?5-class overlay
        axes[row, 2].imshow(img, cmap="gray")
        axes[row, 2].imshow(mask_to_rgb5(e_pred), alpha=0.5)
        e_counts = ','.join(f'{CLASS_NAMES[c][:4]}={(e_pred==c).sum()}' for c in range(1,5) if (e_pred==c).any()) or 'all bg'
        axes[row, 2].set_title(f"EUPE\n{e_counts}", fontsize=8)
        axes[row, 2].axis("off")
    plt.tight_layout(pad=1.0)
    if SAVE_FIG:
        save_dir = REPO_ROOT / "artifacts" / "custom_pick"
        save_dir.mkdir(parents=True, exist_ok=True)
        fname = save_dir / f"pick_{PICK_STEMS[0]}.png"
        fig.savefig(str(fname), dpi=150, bbox_inches="tight", facecolor="white")
        print(f"Saved: {fname}")
    plt.show()
    # Stats: 5-class
    print()
    hdr = "{:12s}  {:>6s} {:>6s} {:>6s} {:>6s}  |  {:>6s} {:>6s} {:>6s} {:>6s}".format(
        "stem", "mod", "sq2", "dark", "br", "mod", "sq2", "dark", "br")
    print(hdr)
    print(f"{'':12s}  {'DINOv3':>24s}  |  {'EUPE':>24s}")
    print("-" * 65)
    for stem, png_path in valid:
        img, dp, ep = load_and_predict(png_path)
        d = [int((dp==c).sum()) for c in range(1,5)]
        e = [int((ep==c).sum()) for c in range(1,5)]
        print("{s:12s}  {m1:6d} {m2:6d} {m3:6d} {m4:6d}  |  {e1:6d} {e2:6d} {e3:6d} {e4:6d}".format(
            s=stem, m1=d[0], m2=d[1], m3=d[2], m4=d[3],
            e1=e[0], e2=e[1], e3=e[2], e4=e[3]))